# 05 Sélection de l'ensemble final

Ce notebook transforme les résultats de benchmark en configuration de production pour l'API.

Il répond à trois objectifs du projet :

1. transformer les benchmarks des notebooks 03 et 04 en une décision finale explicable ;
2. vérifier si le vote doux apporte un gain mesurable face au meilleur modèle seul ;
3. produire `models/ensemble_config.json`, la configuration lue par `src/api/model_loader.py`.

L'idée reste volontairement simple : les CSV locaux sont la source de vérité, la sélection finale garde les trois meilleurs modèles avec une diversité raisonnable, puis le gain du vote doux est mesuré après rechargement des checkpoints.


## Choix méthodologiques

- Les fichiers CSV locaux dans `models/` sont la source principale de vérité. MLflow/DagsHub reste utile pour l'audit, mais le notebook doit rester exécutable même sans connexion.
- Les modèles de screening ne sont pas utilisés pour la configuration finale. On attend les modèles fine-tunés, plus représentatifs de la performance finale.
- Le score de sélection combine plusieurs signaux : F1 macro, balanced accuracy, log loss, temps d'inférence et signes d'overfit.
- Les métriques OOD ne sont utilisées pour sélectionner que si la couverture PlantDoc est fiable pour la tâche. Elles sont toujours affichées comme diagnostic.
- La diversité d'architecture est souhaitable mais pas forcée aveuglément : on limite à deux modèles par famille, sauf si les résultats disponibles ne permettent pas mieux.
- L'architecture finale reste homogène : chaque tâche utilise trois modèles sélectionnés et un vote doux. Le gain face au meilleur modèle seul est mesuré et discuté dans le rapport, mais il ne remplace pas automatiquement l'ensemble.
- Le notebook est prévu pour être lancé après la fin du notebook 04. Il affiche l'état d'avancement, puis s'arrête clairement si une tâche n'a pas assez de modèles fine-tunés.


In [ ]:
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    top_k_accuracy_score,
)

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.models.build import densenet_preprocess_input, resnet_v2_preprocess_input

REPO_ROOT

## Configuration

Les chemins suivent exactement la structure générée par les notebooks précédents. `EXPECTED_TASKS` sert aussi de checklist : si une tâche n'a pas encore de `finetune_results.csv`, elle restera marquée comme incomplète.

In [ ]:
SEED = 42
IMAGE_SIZE = 224
EVAL_BATCH_SIZE = 4
PREFETCH_BUFFER_SIZE = 1
ENSEMBLE_SIZE = 3
REQUIRE_COMPLETE_RESULTS = True
MAX_MODELS_PER_FAMILY = 2
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

DATA_ROOT = REPO_ROOT / "data" / "processed"
OOD_ROOT = REPO_ROOT / "data" / "test_ood"
MODEL_ROOT = REPO_ROOT / "models"
DISEASE_MODEL_ROOT = MODEL_ROOT / "diseases"
ENSEMBLE_OUTPUT_DIR = MODEL_ROOT / "ensemble"
EVALUATION_OUTPUT_DIR = ENSEMBLE_OUTPUT_DIR / "evaluations"
ENSEMBLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPECIES_CONFIGS = {
    "tomato": {"display_name": "Tomate", "expected_classes": 10},
    "apple": {"display_name": "Pommier", "expected_classes": 4},
    "grape": {"display_name": "Vigne", "expected_classes": 4},
    "corn": {"display_name": "Mais", "expected_classes": 4},
    "potato": {"display_name": "Pomme de terre", "expected_classes": 3},
    "pepper": {"display_name": "Poivron", "expected_classes": 2},
    "strawberry": {"display_name": "Fraisier", "expected_classes": 2},
}

EXPECTED_TASKS = ["species", *SPECIES_CONFIGS.keys()]

FINAL_RANKING_METRICS_ID = [
    ("eval_test_f1_macro", 0.30, True),
    ("eval_test_balanced_accuracy", 0.25, True),
    ("eval_val_f1_macro", 0.15, True),
    ("eval_test_log_loss", 0.10, False),
    ("eval_test_ms_per_image", 0.05, False),
    ("loss_gap", 0.05, False),
    ("overfit_gap", 0.05, False),
]

FINAL_RANKING_METRICS_OOD_RELIABLE = [
    ("eval_ood_f1_macro", 0.15, True),
    ("eval_ood_balanced_accuracy", 0.10, True),
]

PREPROCESSOR_CUSTOM_OBJECTS = {
    "DenseNet121": {
        "preprocess_input": tf.keras.applications.densenet.preprocess_input,
        "densenet_preprocess_input": densenet_preprocess_input,
        "plant_disease>densenet_preprocess_input": densenet_preprocess_input,
    },
    "DenseNet169": {
        "preprocess_input": tf.keras.applications.densenet.preprocess_input,
        "densenet_preprocess_input": densenet_preprocess_input,
        "plant_disease>densenet_preprocess_input": densenet_preprocess_input,
    },
    "ResNet50V2": {
        "preprocess_input": tf.keras.applications.resnet_v2.preprocess_input,
        "resnet_v2_preprocess_input": resnet_v2_preprocess_input,
        "plant_disease>resnet_v2_preprocess_input": resnet_v2_preprocess_input,
    },
    "ResNet101V2": {
        "preprocess_input": tf.keras.applications.resnet_v2.preprocess_input,
        "resnet_v2_preprocess_input": resnet_v2_preprocess_input,
        "plant_disease>resnet_v2_preprocess_input": resnet_v2_preprocess_input,
    },
}

np.random.seed(SEED)
tf.random.set_seed(SEED)

## Charger les résultats des notebooks 03 et 04

Chaque ligne correspond à un modèle entraîné. On garde les checkpoints locaux, les métriques et les informations utiles pour reconstruire l'ordre des classes.

In [ ]:
def checkpoint_exists(value: object) -> bool:
    if pd.isna(value):
        return False
    path = Path(str(value))
    if path.exists():
        return True
    if not path.is_absolute() and (REPO_ROOT / path).exists():
        return True
    return False


def resolve_checkpoint_path(value: object) -> Path:
    path = Path(str(value))
    if path.exists():
        return path
    if not path.is_absolute():
        candidate = REPO_ROOT / path
        if candidate.exists():
            return candidate
    return path


def class_names_from_directory(directory: Path) -> list[str]:
    if not directory.exists():
        return []
    return sorted(path.name for path in directory.iterdir() if path.is_dir())


def parse_class_names(value: object, fallback: list[str]) -> list[str]:
    if isinstance(value, str) and value.strip():
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list) and parsed:
                return [str(item) for item in parsed]
        except json.JSONDecodeError:
            pass
    return fallback


def load_task_results(task: str) -> pd.DataFrame:
    if task == "species":
        results_path = MODEL_ROOT / "species" / "finetune_results.csv"
        task_type = "species"
        display_name = "Espèce"
        fallback_class_names = class_names_from_directory(DATA_ROOT / "species" / "train")
    else:
        results_path = DISEASE_MODEL_ROOT / task / "finetune_results.csv"
        task_type = "disease"
        display_name = SPECIES_CONFIGS[task]["display_name"]
        fallback_class_names = class_names_from_directory(DATA_ROOT / task / "train")

    if not results_path.exists():
        return pd.DataFrame()

    frame = pd.read_csv(results_path)
    if frame.empty:
        return frame

    frame = frame.copy()
    frame["task"] = task
    frame["task_type"] = task_type
    frame["display_name"] = frame.get("display_name", display_name)
    frame["results_path"] = str(results_path)
    frame["checkpoint_exists"] = frame["checkpoint_path"].map(checkpoint_exists)
    frame["class_names"] = frame.get("class_names", pd.Series([None] * len(frame))).map(
        lambda value: json.dumps(parse_class_names(value, fallback_class_names))
    )
    frame["num_classes"] = frame["class_names"].map(lambda value: len(json.loads(value)))
    if "ood_reliable" not in frame.columns:
        frame["ood_reliable"] = False
    return frame


def load_all_results() -> pd.DataFrame:
    frames = [load_task_results(task) for task in EXPECTED_TASKS]
    frames = [frame for frame in frames if not frame.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


all_results = load_all_results()
summary_columns = [
    "task", "run_name", "architecture", "architecture_family", "stage",
    "checkpoint_exists", "eval_val_f1_macro", "eval_test_f1_macro",
    "eval_ood_f1_macro", "ood_reliable", "checkpoint_path",
]
display(all_results[[column for column in summary_columns if column in all_results.columns]])

## État d'avancement des tâches

Cette cellule sert de tableau de bord. Comme ce notebook doit être lancé après le notebook 04, chaque tâche doit idéalement avoir trois checkpoints fine-tunés utilisables.

In [ ]:
def readiness_report(results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for task in EXPECTED_TASKS:
        task_results = results[results["task"] == task] if not results.empty else pd.DataFrame()
        usable = task_results[task_results.get("checkpoint_exists", False) == True] if not task_results.empty else task_results
        rows.append(
            {
                "task": task,
                "available_finetuned_runs": int(len(task_results)),
                "usable_checkpoints": int(len(usable)),
                "ready_for_ensemble": bool(len(usable) >= ENSEMBLE_SIZE),
                "ready_for_single_model_check": bool(len(usable) >= 1),
            }
        )
    return pd.DataFrame(rows)


readiness = readiness_report(all_results)
display(readiness)

if REQUIRE_COMPLETE_RESULTS:
    incomplete_tasks = readiness.loc[~readiness["ready_for_ensemble"], "task"].tolist()
    if incomplete_tasks:
        raise RuntimeError(
            "Résultats incomplets pour le notebook 05. "
            f"Tâches à terminer dans le notebook 04 ou 03: {incomplete_tasks}"
        )

## Score de sélection

Le score est un rang pondéré. Il évite de choisir un modèle uniquement parce qu'il a gagné d'un cheveu sur la validation, tout en pénalisant les modèles lents, instables ou trop overfités.

Pour l'OOD, on applique une règle prudente : si PlantDoc ne couvre pas correctement les classes d'une tâche, les métriques OOD restent dans le tableau mais ne pèsent pas dans le score final.

In [ ]:
def bool_series(values: pd.Series) -> pd.Series:
    return values.astype(str).str.lower().isin(["true", "1", "yes"])


def ranking_metric_specs(task_results: pd.DataFrame) -> list[tuple[str, float, bool]]:
    metric_specs = list(FINAL_RANKING_METRICS_ID)
    if "ood_reliable" in task_results.columns and bool_series(task_results["ood_reliable"]).all():
        metric_specs.extend(FINAL_RANKING_METRICS_OOD_RELIABLE)
    return metric_specs


def add_ranking_score(
    summary: pd.DataFrame,
    metric_specs: list[tuple[str, float, bool]],
) -> pd.DataFrame:
    if summary.empty:
        return summary

    ranked = summary.copy()
    score_columns = []
    for metric, weight, higher_is_better in metric_specs:
        if metric not in ranked.columns:
            continue
        values = pd.to_numeric(ranked[metric], errors="coerce")
        if values.notna().sum() == 0:
            continue
        fill_value = values.min() if higher_is_better else values.max()
        score_column = f"score_{metric}"
        ranked[score_column] = values.fillna(fill_value).rank(
            method="average",
            pct=True,
            ascending=higher_is_better,
        ) * weight
        score_columns.append(score_column)

    if not score_columns:
        ranked["ranking_score"] = 0.0
    else:
        ranked["ranking_score"] = ranked[score_columns].sum(axis=1)

    secondary = "eval_test_f1_macro" if "eval_test_f1_macro" in ranked.columns else "eval_val_f1_macro"
    ranked = ranked.sort_values(["ranking_score", secondary], ascending=[False, False])
    ranked["ranking_position"] = np.arange(1, len(ranked) + 1)
    return ranked


def rank_task_results(task_results: pd.DataFrame) -> pd.DataFrame:
    usable = task_results[task_results["checkpoint_exists"]].copy()
    if usable.empty:
        return usable
    return add_ranking_score(usable, ranking_metric_specs(usable))


ranked_results = []
for task, task_results in all_results.groupby("task") if not all_results.empty else []:
    ranked_results.append(rank_task_results(task_results))
ranked_results = pd.concat(ranked_results, ignore_index=True) if ranked_results else pd.DataFrame()

ranking_columns = [
    "task", "ranking_position", "run_name", "architecture", "architecture_family",
    "ranking_score", "eval_test_f1_macro", "eval_test_balanced_accuracy",
    "eval_ood_f1_macro", "eval_test_ms_per_image", "overfit_gap", "ood_reliable",
]
display(ranked_results[[column for column in ranking_columns if column in ranked_results.columns]])

## Comparer les stratégies de sélection

On compare trois options avant de choisir :

- `top3_brut` : les trois meilleurs modèles selon le score ;
- `top3_max2_family` : les trois meilleurs avec au maximum deux modèles par famille ;
- `one_per_family` : une diversité stricte, avec au maximum un modèle par famille.

Pour ce projet, on retient `top3_max2_family`. C'est un compromis simple : il évite les ensembles trop homogènes sans remplacer un très bon modèle par un modèle nettement moins bon uniquement pour varier l'architecture.


In [ ]:
def select_top_models(ranked: pd.DataFrame, *, top_k: int = ENSEMBLE_SIZE) -> pd.DataFrame:
    if ranked.empty:
        return ranked
    selected = ranked.head(top_k).copy()
    selected["selected_rank"] = np.arange(1, len(selected) + 1)
    return selected


def select_diverse_models(
    ranked: pd.DataFrame,
    *,
    top_k: int = ENSEMBLE_SIZE,
    max_per_family: int = MAX_MODELS_PER_FAMILY,
) -> pd.DataFrame:
    if ranked.empty:
        return ranked

    selected_indices = []
    family_counts: dict[str, int] = {}
    for row_index, row in ranked.iterrows():
        family = row.get("architecture_family", "unknown")
        if family_counts.get(family, 0) >= max_per_family:
            continue
        selected_indices.append(row_index)
        family_counts[family] = family_counts.get(family, 0) + 1
        if len(selected_indices) == top_k:
            break

    for row_index, _ in ranked.iterrows():
        if len(selected_indices) == top_k:
            break
        if row_index not in selected_indices:
            selected_indices.append(row_index)

    selected = ranked.loc[selected_indices].copy()
    selected["selected_rank"] = np.arange(1, len(selected) + 1)
    return selected


def summarize_selection_strategy(task: str, strategy_name: str, selected: pd.DataFrame) -> dict[str, object]:
    return {
        "task": task,
        "strategy": strategy_name,
        "model_count": int(len(selected)),
        "families": " + ".join(selected.get("architecture_family", pd.Series(dtype=str)).astype(str).tolist()),
        "architectures": " + ".join(selected.get("architecture", pd.Series(dtype=str)).astype(str).tolist()),
        "mean_test_f1_macro": float(selected["eval_test_f1_macro"].mean()) if "eval_test_f1_macro" in selected else np.nan,
        "min_test_f1_macro": float(selected["eval_test_f1_macro"].min()) if "eval_test_f1_macro" in selected else np.nan,
        "mean_test_log_loss": float(selected["eval_test_log_loss"].mean()) if "eval_test_log_loss" in selected else np.nan,
        "mean_test_ms_per_image": float(selected["eval_test_ms_per_image"].mean()) if "eval_test_ms_per_image" in selected else np.nan,
        "mean_ood_f1_macro": float(selected["eval_ood_f1_macro"].mean()) if "eval_ood_f1_macro" in selected else np.nan,
    }


strategy_rows = []
selected_frames = []

for task in EXPECTED_TASKS:
    task_ranked = ranked_results[ranked_results["task"] == task] if not ranked_results.empty else pd.DataFrame()
    top3_selection = select_top_models(task_ranked)
    max2_selection = select_diverse_models(task_ranked, max_per_family=2)
    strict_family_selection = select_diverse_models(task_ranked, max_per_family=1)

    strategy_rows.extend(
        [
            summarize_selection_strategy(task, "top3_brut", top3_selection),
            summarize_selection_strategy(task, "top3_max2_family", max2_selection),
            summarize_selection_strategy(task, "one_per_family", strict_family_selection),
        ]
    )
    selected_frames.append(max2_selection)

strategy_comparison = pd.DataFrame(strategy_rows)
strategy_comparison_path = ENSEMBLE_OUTPUT_DIR / "selection_strategy_comparison.csv"
strategy_comparison.to_csv(strategy_comparison_path, index=False)

selected_models = pd.concat(selected_frames, ignore_index=True) if selected_frames else pd.DataFrame()
selection_path = ENSEMBLE_OUTPUT_DIR / "selection_summary.csv"
selected_models.to_csv(selection_path, index=False)

selection_columns = [
    "task", "selected_rank", "run_name", "architecture", "architecture_family",
    "ranking_score", "eval_test_f1_macro", "eval_ood_f1_macro", "checkpoint_path",
]
display(strategy_comparison)
display(selected_models[[column for column in selection_columns if column in selected_models.columns]])
print(f"Comparaison des stratégies sauvegardée dans: {strategy_comparison_path}")
print(f"Sélection candidate sauvegardée dans: {selection_path}")


## Fonctions d'évaluation du vote doux

Le vote doux consiste à moyenner les probabilités de chaque modèle :

`proba_ensemble = moyenne(proba_modèle_1, proba_modèle_2, proba_modèle_3)`

On charge les modèles un par un pour éviter de garder trop de checkpoints en mémoire.

In [ ]:
def image_path_records(root: Path, class_names: list[str]) -> pd.DataFrame:
    records = []
    for class_index, class_name in enumerate(class_names):
        class_dir = root / class_name
        if not class_dir.exists():
            continue
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append(
                    {
                        "image_path": str(image_path),
                        "class_index": class_index,
                        "class_name": class_name,
                    }
                )
    return pd.DataFrame(records)


def load_raw_image(image_path: tf.Tensor, label: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    return image, label


def build_path_dataset(root: Path, class_names: list[str]) -> tuple[tf.data.Dataset | None, pd.DataFrame]:
    records = image_path_records(root, class_names)
    if records.empty:
        return None, records
    dataset = tf.data.Dataset.from_tensor_slices(
        (
            records["image_path"].to_numpy(),
            records["class_index"].to_numpy(dtype="int32"),
        )
    )
    dataset = dataset.map(load_raw_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(EVAL_BATCH_SIZE).prefetch(PREFETCH_BUFFER_SIZE)
    return dataset, records


def task_dataset_roots(task: str) -> dict[str, Path]:
    if task == "species":
        return {
            "test": DATA_ROOT / "species" / "test",
            "ood": OOD_ROOT,
        }
    return {
        "test": DATA_ROOT / task / "test",
        "ood": OOD_ROOT / task,
    }


def build_eval_datasets(task: str, class_names: list[str]) -> dict[str, tuple[tf.data.Dataset | None, pd.DataFrame]]:
    roots = task_dataset_roots(task)
    return {name: build_path_dataset(root, class_names) for name, root in roots.items()}


def load_checkpoint_model(architecture: str, checkpoint_path: Path) -> tf.keras.Model:
    return tf.keras.models.load_model(
        str(checkpoint_path),
        compile=False,
        safe_mode=False,
        custom_objects=PREPROCESSOR_CUSTOM_OBJECTS.get(architecture, {}),
    )


def predict_dataset(model: tf.keras.Model, dataset: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray, float]:
    y_true = []
    y_score_batches = []
    image_count = 0
    started_at = time.perf_counter()
    for images, labels in dataset:
        predictions = model.predict(images, verbose=0)
        y_score_batches.append(predictions)
        y_true.extend(labels.numpy().astype(int).tolist())
        image_count += int(images.shape[0])
    elapsed = time.perf_counter() - started_at
    return np.array(y_true, dtype="int32"), np.concatenate(y_score_batches, axis=0), elapsed


def classification_metrics(
    y_true: np.ndarray,
    y_score: np.ndarray,
    *,
    num_classes: int,
    elapsed_seconds: float,
) -> dict[str, float]:
    y_pred = np.argmax(y_score, axis=1)
    labels = np.arange(num_classes)
    image_count = max(1, len(y_true))
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "log_loss": float(log_loss(y_true, y_score, labels=labels)),
        "ms_per_image": float((elapsed_seconds / image_count) * 1000),
        "image_count": float(image_count),
    }
    top_k = min(2, num_classes - 1) if num_classes > 2 else 1
    if top_k > 1:
        metrics[f"top{top_k}_accuracy"] = float(top_k_accuracy_score(y_true, y_score, k=top_k, labels=labels))
    return metrics


def predict_checkpoint(row: pd.Series, dataset: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray, float]:
    tf.keras.backend.clear_session()
    model = None
    try:
        model = load_checkpoint_model(row["architecture"], resolve_checkpoint_path(row["checkpoint_path"]))
        return predict_dataset(model, dataset)
    finally:
        if model is not None:
            del model
        tf.keras.backend.clear_session()
        gc.collect()


def evaluate_single_and_ensemble(
    task: str,
    selected: pd.DataFrame,
    dataset_name: str,
    dataset: tf.data.Dataset,
    num_classes: int,
) -> list[dict[str, object]]:
    rows = []
    ensemble_score_sum = None
    ensemble_elapsed = 0.0
    reference_y_true = None

    for _, model_row in selected.iterrows():
        y_true, y_score, elapsed = predict_checkpoint(model_row, dataset)
        reference_y_true = y_true if reference_y_true is None else reference_y_true
        if not np.array_equal(reference_y_true, y_true):
            raise ValueError(f"Ordre des labels incohérent pendant l'évaluation de {task}/{dataset_name}.")

        single_metrics = classification_metrics(
            y_true,
            y_score,
            num_classes=num_classes,
            elapsed_seconds=elapsed,
        )
        rows.append(
            {
                "task": task,
                "dataset": dataset_name,
                "model_type": "single",
                "run_name": model_row["run_name"],
                "architecture": model_row["architecture"],
                **single_metrics,
            }
        )

        ensemble_score_sum = y_score if ensemble_score_sum is None else ensemble_score_sum + y_score
        ensemble_elapsed += elapsed

    if ensemble_score_sum is not None and len(selected) >= 2:
        ensemble_score = ensemble_score_sum / len(selected)
        ensemble_metrics = classification_metrics(
            reference_y_true,
            ensemble_score,
            num_classes=num_classes,
            elapsed_seconds=ensemble_elapsed,
        )
        rows.append(
            {
                "task": task,
                "dataset": dataset_name,
                "model_type": "ensemble_soft_vote",
                "run_name": "+".join(selected["run_name"].tolist()),
                "architecture": "+".join(selected["architecture"].tolist()),
                **ensemble_metrics,
            }
        )
    return rows

## Évaluer le gain de l'ensemble, tâche par tâche

L'évaluation recharge plusieurs checkpoints Keras et peut être coûteuse en mémoire. Pour éviter de faire planter WSL, on ne lance plus toutes les tâches dans une seule cellule.

Principe :

1. exécuter la cellule de fonctions ci-dessous ;
2. exécuter une seule cellule de tâche à la fois ;
3. attendre la fin et vérifier le CSV sauvegardé ;
4. passer à la tâche suivante.

Chaque tâche écrit immédiatement son résultat dans `models/ensemble/evaluations/<task>_ensemble_evaluation.csv`. La synthèse globale recharge ensuite ces fichiers.


In [ ]:
def task_evaluation_path(task: str) -> Path:
    return EVALUATION_OUTPUT_DIR / f"{task}_ensemble_evaluation.csv"


def load_saved_evaluations() -> pd.DataFrame:
    frames = []
    for evaluation_file in sorted(EVALUATION_OUTPUT_DIR.glob("*_ensemble_evaluation.csv")):
        frame = pd.read_csv(evaluation_file)
        frame["evaluation_file"] = str(evaluation_file)
        frames.append(frame)

    evaluation = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not evaluation.empty:
        evaluation_path = ENSEMBLE_OUTPUT_DIR / "ensemble_evaluation.csv"
        evaluation.to_csv(evaluation_path, index=False)
    return evaluation


def evaluate_task_ensemble(task: str, datasets_to_run: tuple[str, ...] = ("test", "ood")) -> pd.DataFrame:
    task_selection = selected_models[selected_models["task"] == task] if not selected_models.empty else pd.DataFrame()
    if len(task_selection) < ENSEMBLE_SIZE:
        raise RuntimeError(
            f"{task}: {len(task_selection)} modèle(s) sélectionné(s), "
            f"{ENSEMBLE_SIZE} requis pour le vote doux."
        )

    class_names = json.loads(task_selection.iloc[0]["class_names"])
    datasets = build_eval_datasets(task, class_names)
    rows = []
    output_path = task_evaluation_path(task)

    print(f"{task}: {len(task_selection)} modèle(s), {len(class_names)} classe(s)")
    print(f"Sauvegarde: {output_path}")

    for dataset_name in datasets_to_run:
        dataset, records = datasets.get(dataset_name, (None, pd.DataFrame()))
        if dataset is None or records.empty:
            print(f"  - {dataset_name}: aucune image disponible")
            continue

        print(f"  - {dataset_name}: {len(records)} images")
        rows.extend(
            evaluate_single_and_ensemble(
                task=task,
                selected=task_selection,
                dataset_name=dataset_name,
                dataset=dataset,
                num_classes=len(class_names),
            )
        )

        partial_evaluation = pd.DataFrame(rows)
        partial_evaluation.to_csv(output_path, index=False)
        print(f"    sauvegardé intermédiaire: {len(partial_evaluation)} ligne(s)")

        del dataset
        gc.collect()
        tf.keras.backend.clear_session()

    task_evaluation = pd.DataFrame(rows)
    if not task_evaluation.empty:
        task_evaluation.to_csv(output_path, index=False)
        display(task_evaluation)
        print(f"Évaluation {task} sauvegardée dans: {output_path}")
    else:
        print(f"{task}: aucune évaluation produite.")

    global evaluation
    evaluation = load_saved_evaluations()
    return task_evaluation


evaluation = load_saved_evaluations()
if evaluation.empty:
    print("Aucune évaluation sauvegardée pour le moment. Lance les cellules par tâche ci-dessous.")
else:
    display(evaluation)
    print(f"Évaluations rechargées depuis: {EVALUATION_OUTPUT_DIR}")


### Évaluation espèce

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [ ]:
evaluate_task_ensemble("species")


### Évaluation tomate

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [ ]:
evaluate_task_ensemble("tomato")


### Évaluation pommier

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [ ]:
evaluate_task_ensemble("apple")


### Évaluation vigne

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [ ]:
evaluate_task_ensemble("grape")


### Évaluation maïs

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [ ]:
evaluate_task_ensemble("corn")


### Évaluation pomme de terre

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [ ]:
evaluate_task_ensemble("potato")


### Évaluation poivron

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [ ]:
evaluate_task_ensemble("pepper")


### Évaluation fraisier

Exécuter cette cellule seule, puis attendre la fin avant de passer à la tâche suivante.

In [ ]:
evaluate_task_ensemble("strawberry")


### Recharger toutes les évaluations sauvegardées

Exécuter cette cellule avant la synthèse si le kernel a été redémarré entre deux tâches.

In [ ]:
evaluation = load_saved_evaluations()
if evaluation.empty:
    print("Aucune évaluation sauvegardée.")
else:
    display(evaluation)
    print(f"Évaluation globale sauvegardée dans: {ENSEMBLE_OUTPUT_DIR / 'ensemble_evaluation.csv'}")


## Synthèse du gain

On compare l'ensemble au meilleur modèle seul observé dans la même évaluation.

Cette mesure sert à documenter l'intérêt réel du vote doux :

- gain positif : l'ensemble améliore la tâche ;
- gain proche de zéro : l'ensemble stabilise la décision sans perte notable ;
- gain négatif : le rapport doit le signaler honnêtement.

Pour garder une architecture claire et cohérente avec le projet, la configuration finale conserve trois modèles par tâche. Le vote doux reste donc la stratégie de production, et les gains sont utilisés comme analyse critique plutôt que comme règle de fallback automatique.


In [ ]:
def gain_summary(evaluation: pd.DataFrame) -> pd.DataFrame:
    if evaluation.empty:
        return evaluation

    rows = []
    for (task, dataset), group in evaluation.groupby(["task", "dataset"]):
        singles = group[group["model_type"] == "single"]
        ensembles = group[group["model_type"] == "ensemble_soft_vote"]
        if singles.empty or ensembles.empty:
            continue

        best_single = singles.sort_values("f1_macro", ascending=False).iloc[0]
        ensemble = ensembles.iloc[0]
        rows.append(
            {
                "task": task,
                "dataset": dataset,
                "best_single": best_single["run_name"],
                "ensemble_models": ensemble["run_name"],
                "best_single_f1_macro": best_single["f1_macro"],
                "ensemble_f1_macro": ensemble["f1_macro"],
                "gain_f1_macro": ensemble["f1_macro"] - best_single["f1_macro"],
                "best_single_accuracy": best_single["accuracy"],
                "ensemble_accuracy": ensemble["accuracy"],
                "gain_accuracy": ensemble["accuracy"] - best_single["accuracy"],
                "best_single_log_loss": best_single["log_loss"],
                "ensemble_log_loss": ensemble["log_loss"],
                "gain_log_loss": best_single["log_loss"] - ensemble["log_loss"],
            }
        )
    return pd.DataFrame(rows)


def build_final_decisions(selected: pd.DataFrame, gains: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for task in EXPECTED_TASKS:
        task_selection = selected[selected["task"] == task] if not selected.empty else pd.DataFrame()
        if task_selection.empty:
            continue

        task_gains = gains[(gains["task"] == task) & (gains["dataset"] == "test")] if not gains.empty else pd.DataFrame()
        gain_f1 = float(task_gains.iloc[0]["gain_f1_macro"]) if not task_gains.empty else None
        gain_log_loss = float(task_gains.iloc[0]["gain_log_loss"]) if not task_gains.empty else None
        rows.append(
            {
                "task": task,
                "final_strategy": "soft_vote_mean_probabilities",
                "model_count": int(len(task_selection)),
                "gain_f1_macro_test": gain_f1,
                "gain_log_loss_test": gain_log_loss,
                "reason": "Architecture d'ensemble conservée : trois modèles sélectionnés votent pour chaque tâche.",
            }
        )
    return pd.DataFrame(rows)


gains = gain_summary(evaluation)
gains_path = ENSEMBLE_OUTPUT_DIR / "ensemble_gain_summary.csv"
if not gains.empty:
    gains.to_csv(gains_path, index=False)
    display(gains)
    print(f"Synthèse sauvegardée dans: {gains_path}")
else:
    print("Pas encore de synthèse de gain.")

final_models = selected_models.copy()
if not final_models.empty:
    final_models["final_strategy"] = "soft_vote_mean_probabilities"
    final_models["final_decision_reason"] = "Architecture d'ensemble conservée : trois modèles sélectionnés votent pour chaque tâche."

final_decisions = build_final_decisions(selected_models, gains)
final_selection_path = ENSEMBLE_OUTPUT_DIR / "final_selection_summary.csv"
final_decisions_path = ENSEMBLE_OUTPUT_DIR / "final_decisions.csv"

if not final_models.empty:
    final_models.to_csv(final_selection_path, index=False)
    final_decisions.to_csv(final_decisions_path, index=False)
    display(final_decisions)
    display(final_models[[column for column in [
        "task", "selected_rank", "final_strategy", "run_name", "architecture",
        "architecture_family", "eval_test_f1_macro", "final_decision_reason",
    ] if column in final_models.columns]])
    print(f"Sélection finale sauvegardée dans: {final_selection_path}")
    print(f"Décisions finales sauvegardées dans: {final_decisions_path}")
else:
    print("Pas encore de sélection finale.")


## Sauvegarder la configuration finale

La configuration est volontairement plus riche qu'une simple liste de noms. Elle contient les chemins de checkpoints, l'ordre des classes, les métriques de sélection et la stratégie finale retenue.

C'est ce fichier que l'API lit ensuite pour charger les bons modèles. Chaque tâche conserve trois modèles et utilise le vote doux, ce qui garde l'architecture de production simple à expliquer et cohérente avec le projet.

Comme `REQUIRE_COMPLETE_RESULTS=True`, le fichier final est écrit dans `models/ensemble_config.json` seulement quand toutes les tâches attendues disposent de trois modèles sélectionnés.


In [ ]:
def row_float(row: pd.Series, key: str) -> float | None:
    value = row.get(key)
    if pd.isna(value):
        return None
    return float(value)


def build_ensemble_config(final_selected: pd.DataFrame, gains: pd.DataFrame, decisions: pd.DataFrame) -> dict[str, object]:
    tasks = {}
    for task, task_selection in final_selected.groupby("task") if not final_selected.empty else []:
        class_names = json.loads(task_selection.iloc[0]["class_names"])
        task_gains = gains[gains["task"] == task] if not gains.empty else pd.DataFrame()
        task_decision = decisions[decisions["task"] == task] if not decisions.empty else pd.DataFrame()
        decision_payload = task_decision.iloc[0].to_dict() if not task_decision.empty else {}

        models = []
        for _, row in task_selection.sort_values("selected_rank").iterrows():
            checkpoint_path = resolve_checkpoint_path(row["checkpoint_path"])
            models.append(
                {
                    "run_name": row["run_name"],
                    "architecture": row["architecture"],
                    "architecture_family": row.get("architecture_family"),
                    "checkpoint_path": str(checkpoint_path),
                    "selected_rank": int(row["selected_rank"]),
                    "ranking_score": row_float(row, "ranking_score"),
                    "eval_test_f1_macro": row_float(row, "eval_test_f1_macro"),
                    "eval_ood_f1_macro": row_float(row, "eval_ood_f1_macro"),
                }
            )

        tasks[task] = {
            "task_type": task_selection.iloc[0]["task_type"],
            "display_name": task_selection.iloc[0].get("display_name", task),
            "strategy": "soft_vote_mean_probabilities",
            "selection_policy": "top3_max2_family",
            "decision_reason": decision_payload.get("reason"),
            "image_size": IMAGE_SIZE,
            "class_names": class_names,
            "models": models,
            "gains": task_gains.to_dict(orient="records"),
        }

    complete_tasks = sorted(task for task, payload in tasks.items() if len(payload["models"]) >= ENSEMBLE_SIZE)
    missing_tasks = sorted(set(EXPECTED_TASKS) - set(complete_tasks))
    return {
        "version": 1,
        "created_by": "notebooks/05_ensemble_selection.ipynb",
        "selection_source": "local CSV files from notebooks 03 and 04",
        "selection_policy": "top3_max2_family",
        "complete": len(missing_tasks) == 0,
        "complete_tasks": complete_tasks,
        "missing_tasks": missing_tasks,
        "tasks": tasks,
    }


ensemble_config = build_ensemble_config(final_models, gains, final_decisions)
if REQUIRE_COMPLETE_RESULTS and not ensemble_config["complete"]:
    raise RuntimeError(f"Configuration incomplète, tâches manquantes: {ensemble_config['missing_tasks']}")

config_path = MODEL_ROOT / "ensemble_config.json" if ensemble_config["complete"] else ENSEMBLE_OUTPUT_DIR / "ensemble_config_partial.json"
config_path.write_text(json.dumps(ensemble_config, indent=2), encoding="utf-8")

print(f"Configuration sauvegardée dans: {config_path}")
print(f"Configuration complète: {ensemble_config['complete']}")
print(f"Tâches prêtes: {ensemble_config['complete_tasks']}")
print(f"Tâches manquantes: {ensemble_config['missing_tasks']}")


## Conclusion à reporter

À la fin de l'exécution complète, reprendre quatre éléments dans le rapport :

1. Le tableau de comparaison des stratégies (`sélection_strategy_comparison.csv`).
2. Le tableau de sélection candidate (`sélection_summary.csv`).
3. Le tableau de gain de l'ensemble (`ensemble_gain_summary.csv`).
4. Le tableau de décision finale (`final_decisions.csv`), qui rappelle que l'architecture de production conserve le vote doux à trois modèles.

La justification principale doit rester simple : le projet privilégie une architecture d'ensemble homogène, avec une sélection mesurée sur les résultats et une diversité raisonnable, sans forcer une architecture moins bonne uniquement pour remplir une contrainte artificielle.
